# [CPSC 322](https://github.com/GonzagaCPSC322) Data Science Algorithms
[Gonzaga University](https://www.gonzaga.edu/)

[Gina Sprint](http://cs.gonzaga.edu/faculty/sprint/)

# PA6 Decision Trees (100 pts)

## Learner Objectives
At the conclusion of this programming assignment, participants should be able to:
* Represent a tree in Python
* Implement a decision tree classifier using the TDIDT algorithm
* Select an attribute using entropy
* Extract rules from a decision tree
* (BONUS) Visualize a tree with Graphviz and DOT


## Prerequisites
Before starting this programming assignment, participants should be able to:
* Implement a $k$NN and Naive Bayes classifier
* Evaluate classifiers using train/test sets
* Understand tree representations and common tree traversal algorithms
* Understand recursion

## Acknowledgments
Content used in this assignment is based upon information in the following sources:
* Dr. Shawn Bowers' Data Mining HW5

## Github Classroom Setup
For this assignment, you will use GitHub Classroom to create a private code repository to track code changes and submit your assignment. Open this PA6 link to accept the assignment and create a private repository for your assignment in Github classroom: https://classroom.github.com/a/5bcVrKHV

Your repo, for example, will be named GonzagaCPSC322/pa6-yourusername (where yourusername is your Github username). I highly recommend committing/pushing regularly so your work is always backed up. We will grade your most recent commit, even if that commit is after the due date (your work will be marked late if this is the case).

## Overview and Requirements
This assignment involves implementing a decision tree classifier. It has two main parts:
1. `mysklearn`: Test and implement general and re-usable classifiers
1. Datasets
    1. Auto Dataset Classification (pa6-auto.ipynb): Write a Jupyter Notebook that uses `mysklearn` to perform data classification tasks on an automobile dataset
    1. Titanic Dataset Classification (pa6-titanic.ipynb): Write a Jupyter Notebook that uses `mysklearn` to perform data classification tasks on a titanic survival dataset

I highly encourage you to design functions that are generic and re-usable for future programming assignments and data mining tasks.

Note: we are learning data science from scratch! The only non-standard Python library you should need to use for this assignment is `tabulate`. For testing purposes, you may use `numpy` and `scipy`. This means that beyond these libraries, you should not `pip install` any additional libraries beyond what is included in the [continuumio/anaconda3:2020.11](https://hub.docker.com/r/continuumio/anaconda3) Docker image and you should not use `pandas/sklearn/`etc... (exceptions are made for testing purposes only!!).

## Part 1: `mysklearn` (60 pts)
Our decision tree classifier we are going to implement for PA6 will:
* Be constructed/stored using the nested list representation described in class
* Use the entropy-based attribute selection method described in class
* Use the majority voting method to deal with clashes
* Only work with categorical attributes (you will not need to use any continuous attributes with your decision tree classifier in this assignment)

### Step 1: Implement Decision Tree Unit Tests for `myclassifiers.py`
Finish the decision tree unit tests in `test_myclassifiers.py` for `MyDecisionTreeClassifier` (Class API design inspiration: Sci-kit Learn's [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html))
1. `fit(X_train, y_train)`
    1. Finish the test function `test_decision_tree_classifier_fit()` by implementing the following test cases
        1. Use the 14 instance "interview" training set example traced in class on the iPad, asserting against the tree constructed in [A Decision Trees Lab Task #4](https://github.com/GonzagaCPSC322/U5-Decision-Trees/blob/master/A%20Decision%20Trees.ipynb)
        1. Use the Bramer 4.1 Figure 4.3 *degrees* dataset example, asserting against the tree you create when you finish Bramer 5.5 Self-assessment exercise 1 (note that Bramer's tree in  Figure 4.4 is NOT the entropy-based solution for this dataset, you will need to finish the exercise to construct the correct tree)
1. `predict(X_test)`
    1. Finish the test function `test_decision_tree_classifier_predict()` by implementing the following test cases
        1. Use the 14 instance "interview" training set example traced in class on the iPad, asserting against the two predictions made in [A Decision Trees Lab Task #5](https://github.com/GonzagaCPSC322/U5-Decision-Trees/blob/master/A%20Decision%20Trees.ipynb)
        1. For the Bramer *degrees* dataset, use the following unseen instances `[["B", "B", "B", "B", "B"], ["A", "A", "A", "A", "A"], ["A", "A", "A", "A", "B"]]`, asserting against the solution predictions from your own desk check
        
For convenience, I've provided the "interview" and Bramer datasets as Python lists below.

In [1]:
# interview dataset
interview_header = ["level", "lang", "tweets", "phd", "interviewed_well"]
interview_table = [
        ["Senior", "Java", "no", "no", "False"],
        ["Senior", "Java", "no", "yes", "False"],
        ["Mid", "Python", "no", "no", "True"],
        ["Junior", "Python", "no", "no", "True"],
        ["Junior", "R", "yes", "no", "True"],
        ["Junior", "R", "yes", "yes", "False"],
        ["Mid", "R", "yes", "yes", "True"],
        ["Senior", "Python", "no", "no", "False"],
        ["Senior", "R", "yes", "no", "True"],
        ["Junior", "Python", "yes", "no", "True"],
        ["Senior", "Python", "yes", "yes", "True"],
        ["Mid", "Python", "no", "yes", "True"],
        ["Mid", "Java", "yes", "no", "True"],
        ["Junior", "Python", "no", "yes", "False"]
    ]

# note: this tree uses the generic "att#" attribute labels because fit() does not and should not accept attribute names
# note: the attribute values are sorted alphabetically
interview_tree = \
        ["Attribute", "att0",
            ["Value", "Junior", 
                ["Attribute", "att3",
                    ["Value", "no", 
                        ["Leaf", "True", 3, 5]
                    ],
                    ["Value", "yes", 
                        ["Leaf", "False", 2, 5]
                    ]
                ]
            ],
            ["Value", "Mid",
                ["Leaf", "True", 4, 14]
            ],
            ["Value", "Senior",
                ["Attribute", "att2",
                    ["Value", "no",
                        ["Leaf", "False", 3, 5]
                    ],
                    ["Value", "yes",
                        ["Leaf", "True", 2, 5]
                    ]
                ]
            ]
        ]

# bramer degrees dataset
degrees_header = ["SoftEng", "ARIN", "HCI", "CSA", "Project", "Class"]
degrees_table = [
        ["A", "B", "A", "B", "B", "SECOND"],
        ["A", "B", "B", "B", "A", "FIRST"],
        ["A", "A", "A", "B", "B", "SECOND"],
        ["B", "A", "A", "B", "B", "SECOND"],
        ["A", "A", "B", "B", "A", "FIRST"],
        ["B", "A", "A", "B", "B", "SECOND"],
        ["A", "B", "B", "B", "B", "SECOND"],
        ["A", "B", "B", "B", "B", "SECOND"],
        ["A", "A", "A", "A", "A", "FIRST"],
        ["B", "A", "A", "B", "B", "SECOND"],
        ["B", "A", "A", "B", "B", "SECOND"],
        ["A", "B", "B", "A", "B", "SECOND"],
        ["B", "B", "B", "B", "A", "SECOND"],
        ["A", "A", "B", "A", "B", "FIRST"],
        ["B", "B", "B", "B", "A", "SECOND"],
        ["A", "A", "B", "B", "B", "SECOND"],
        ["B", "B", "B", "B", "B", "SECOND"],
        ["A", "A", "B", "A", "A", "FIRST"],
        ["B", "B", "B", "A", "A", "SECOND"],
        ["B", "B", "A", "A", "B", "SECOND"],
        ["B", "B", "B", "B", "A", "SECOND"],
        ["B", "A", "B", "A", "B", "SECOND"],
        ["A", "B", "B", "B", "A", "FIRST"],
        ["A", "B", "A", "B", "B", "SECOND"],
        ["B", "A", "B", "B", "B", "SECOND"],
        ["A", "B", "B", "B", "B", "SECOND"],
    ]

degrees_tree = [] # TODO: figure out what this is by finishing Bramer 5.5 Self-assessment exercise 1

### Step 2: `fit()` and `predict()`
Complete the `mysklearn.myclassifiers.MyDecisionTreeClassifier` methods `fit()` and `predict()` and test your code for functional correctness against the above unit tests.

### Step 3: `print_decision_rules()`
Finish the `print_decision_rules()` method of `MyDecisionTreeClassifier` that prints out the rules inferred from a decision tree created from a call to `fit()`. Your rules should take the form:

```
IF att0 == val AND ... THEN class = label
IF att1 == val AND ... THEN class = label
...
```

## Part 2: Datasets (30 pts)

### Step 1: 🚗 Auto Classification 🚗
Create a decision tree classifier that predicts mpg rankings (given in PA2) from the cylinders, weight, and model year attributes. Discretize weights using the NHTSA vehicle sizes from PA5:

|Ranking |Range|
|-|-|
|5 |$\geq$ 3500
|4 |3000-3499|
|3 |2500-2999|
|2 |2000-2499|
|1 |$\leq$ 1999|

* Test your classifier using stratified k-fold cross-validation (with k = 10), and compare your results to those from PA5
* Create a confusion matrix for the result and format your results as per PA4 and PA5
* Print out the rules inferred from your decision tree classifiers when run over the entire dataset (as opposed to the cross validation trees)
    * Based on the rules, determine ways your trees can/should be pruned. Note you do not need to write code to perform pruning, just explain how they can be pruned and give the resulting "pruned" rules

### Step 2: 🚢 Titanic Classification 🚢 
Create a decision tree classifier for the titanic dataset. Since each attribute is categorical, you do not need to perform any discretization, etc. 
* Test your classifier using stratified k-fold cross-validation (with k = 10), and compare your results to those from PA5
* Create a confusion matrix for the result and format your results as per PA4 and PA5
* Print out the rules inferred from your decision tree classifiers when run over the entire dataset (as opposed to the cross validation trees)
    * Based on the rules, determine ways your trees can/should be pruned. Note you do not need to write code to perform pruning, just explain how they can be pruned and give the resulting "pruned" rules

## BONUS (5 pts)
Finish the `visualize_tree()` method of `MyDecisionTreeClassifier` that generates a Graphviz .dot file and a .pdf file. The .dot file is used to produce a visual PDF representation of the decision tree. The tree should have unique nodes for all attributes and leaves in the tree (this means no node should have more than one incoming edge in the visualization). You can read more about dot and how to do this in the [D Tree Visualization and Pruning](https://github.com/GonzagaCPSC322/U5-Decision-Trees/blob/master/D%20Tree%20Visualization%20and%20Pruning.ipynb) notes on Github. Call this method for the trees fit over the entire datasets for both part 2 steps 1 and 2 (e.g. the trees used to print the decision rules for each dataset). 

Include the graphs generated as PDF files (produced via dot) in your repo in a directory called `tree_vis`.

## Submitting Assignments
1. Use Github classroom to submit your assignment via a Github repo. See the "Github Classroom Setup" section at the beginning of this document for details on how to do this. You must commit your solution by the due date and time.
1. Double check your submission by "pretending to be the grader": clone (or download a zip) your submission repo and run your code in a fresh [continuumio/anaconda3:2020.11](https://hub.docker.com/r/continuumio/anaconda3) Docker container like we will when we grade your code.

## Grading Guidelines
This assignment is worth 100 points + 5 points bonus. Your assignment will be evaluated based on a successful execution in the [continuumio/anaconda3:2020.11](https://hub.docker.com/r/continuumio/anaconda3) Docker container and adherence to the program requirements. We will grade according to the following criteria:
* 15 pts for correct part 1 step 1 (finish Bramer 5.5 Self-assessment exercise 1 and define unit tests)
* 35 pts for correct part 1 step 2 (finish `MyDecisionTreeClassifier` and pass tests)
* 10 pts for correct part 1 step 3 (finish `print_decision_rules()`)
* 15 pts for correct part 2 step 1 (auto data classification)
* 15 pts for correct part 2 step 2 (titanic classification)
* 10 pts for adherence to course [coding standard](https://nbviewer.jupyter.org/github/GonzagaCPSC322/PAs/blob/master/Coding%20Standard.ipynb), including data storytelling (narrative is clear and grammatically correct, Notebook is organized with headers, formulas are typeset with Latex, etc.).

<img src="https://miro.medium.com/max/512/0*dknuIMqtCrKXBPNN" width="400"/>